In [29]:
import os
import cv2 as cv
import numpy as np
import rasterio
from pathlib import Path
from tqdm import tqdm
from image_processor import ImageProcessor, Bands, ThreeBandInputPaths



In [30]:


INPUT_DIR_LARGE = Path("/home/samuel/SDU/MasterThesis/Orthomosaics/large/processed_output")
INPUT_DIR_MID = Path("/home/samuel/SDU/MasterThesis/Orthomosaics/mid/processed_output")
INPUT_DIR_SMALL = Path("/home/samuel/SDU/MasterThesis/Orthomosaics/small/processed_output")

OUTPUT_DIR = Path("/home/samuel/SDU/MasterThesis/Orthomosaics/band_combinations") / "float_combinations"
if not OUTPUT_DIR.exists():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR_LARGE = OUTPUT_DIR / "large"
OUTPUT_DIR_MID = OUTPUT_DIR / "mid"
OUTPUT_DIR_SMALL = OUTPUT_DIR / "small"

# PATHS = [[INPUT_DIR_LARGE, OUTPUT_DIR_LARGE], [INPUT_DIR_MID, OUTPUT_DIR_MID], [INPUT_DIR_SMALL, OUTPUT_DIR_SMALL]]

PATHS = [[INPUT_DIR_LARGE, OUTPUT_DIR_LARGE],[INPUT_DIR_MID, OUTPUT_DIR_MID], [INPUT_DIR_SMALL, OUTPUT_DIR_SMALL]]
# if not OUTPUT_DIR_LARGE.exists():
#     OUTPUT_DIR_LARGE.mkdir(parents=True, exist_ok=True)
if not OUTPUT_DIR_MID.exists():
    OUTPUT_DIR_MID.mkdir(parents=True, exist_ok=True)
if not OUTPUT_DIR_SMALL.exists():
    OUTPUT_DIR_SMALL.mkdir(parents=True, exist_ok=True)

proc = ImageProcessor(INPUT_DIR_MID, OUTPUT_DIR_LARGE, "")

In [31]:
# NEN combination (nir, red, ngrdi)
ending = "NEN"

for index, path in enumerate(PATHS):

    input_dir = path[0]
    OUTPUT_DIR = path[1]
    dir = Path(str(OUTPUT_DIR) + "/" + ending)

    if not dir.exists():
        proc.set_input_path(input_dir / "image_tiles")

        for img_name in tqdm(sorted(os.listdir(proc.input_path)), desc="NEN image creation"):
            if "xml" in img_name:
                continue
            img_name_nir = img_name.replace(".tif", "_NIR.tif")
            img_name_er = img_name.replace(".tif", "_EXTEND_RED.tif")
            img_name_eg = img_name.replace(".tif", "_ngrdi.tif")
            proc.calculate_three_band_image(
                input_paths = ThreeBandInputPaths(
                    band1=input_dir / "nir" / img_name_nir,
                    band2=input_dir / "extended_red" / img_name_er,
                    band3=input_dir / "image_tiles_indeces" / "NGRDI" / img_name_eg,
                ),
                output_path=dir,
                ending=ending
            )

    else:
        print("NEN image creation skipped; output directory already exists.")

NEN image creation skipped; output directory already exists.
NEN image creation skipped; output directory already exists.


NEN image creation: 100%|██████████| 391/391 [00:20<00:00, 18.99it/s]


In [32]:
# NRG combination (nir, red, green)
ending = "NRG"

for index, path in enumerate(PATHS):

    input_dir = path[0]
    OUTPUT_DIR = path[1]
    dir = Path(str(OUTPUT_DIR) + "/" + ending)

    if not dir.exists():
        percentiles_nir = proc.collect_percentiles(input_dir / "nir", lower=2, upper=98)
        percentiles_er = proc.collect_percentiles(input_dir / "extended_red", lower=2, upper=98)
        percentiles_eg = proc.collect_percentiles(input_dir / "extended_green", lower=2, upper=98)
        print("Percentiles NIR:", percentiles_nir)
        print("Percentiles EXTENDED RED:", percentiles_er)
        print("Percentiles EXTENDED GREEN:", percentiles_eg)
        percentiles = [percentiles_nir, percentiles_er, percentiles_eg]

    if not dir.exists():
        proc.set_input_path(input_dir / "image_tiles")

        for img_name in tqdm(sorted(os.listdir(proc.input_path)), desc="NEN image creation"):
            if "xml" in img_name:
                continue
            img_name_nir = img_name.replace(".tif", "_NIR.tif")
            img_name_er = img_name.replace(".tif", "_EXTEND_RED.tif")
            img_name_eg = img_name.replace(".tif", "_EXTEND_GREEN.tif")

            proc.calculate_three_band_image(
                input_paths = ThreeBandInputPaths(
                    band1=input_dir / "extended_red" / img_name_er,
                    band2=input_dir /  "extended_green" / img_name_eg,
                    band3=input_dir / "nir" / img_name_nir,
                ),
                output_path=dir,
                ending=ending,
                percentiles=[]
            )

    else:
        print("NRG image creation skipped; output directory already exists.")

NRG image creation skipped; output directory already exists.


100%|██████████| 425/425 [00:14<00:00, 28.34it/s]


Percentiles NIR: (np.float32(2741.0), np.float32(65535.0))
Percentiles EXTENDED RED: (np.float32(1325.0), np.float32(65535.0))
Percentiles EXTENDED GREEN: (np.float32(994.0), np.float32(65535.0))


100%|██████████| 391/391 [00:13<00:00, 29.73it/s]


Percentiles NIR: (np.float32(0.0), np.float32(255.0))
Percentiles EXTENDED RED: (np.float32(0.0), np.float32(255.0))
Percentiles EXTENDED GREEN: (np.float32(0.0), np.float32(255.0))


NEN image creation: 100%|██████████| 391/391 [00:15<00:00, 24.58it/s]


In [33]:
# NRD combination (nir, red, ndvi)
ending = "NRD"

for index, path in enumerate(PATHS):

    input_dir = path[0]
    OUTPUT_DIR = path[1]
    dir = Path(str(OUTPUT_DIR) + "/" + ending)

    if not dir.exists():
        proc.set_input_path(input_dir / "image_tiles")

        for img_name in tqdm(sorted(os.listdir(proc.input_path)), desc="NEN image creation"):
            if "xml" in img_name:
                continue
            img_name_nir = img_name.replace(".tif", "_NIR.tif")
            img_name_er = img_name.replace(".tif", "_EXTEND_RED.tif")
            img_name_ndvi = img_name.replace(".tif", "_ndvi.tif")
            proc.calculate_three_band_image(
                input_paths = ThreeBandInputPaths(
                    band2=input_dir / "nir" / img_name_nir,
                    band1=input_dir / "extended_red" / img_name_er,
                    band3=input_dir / "image_tiles_indeces" / "NDVI" / img_name_ndvi,
                ),
                output_path=dir,
                ending=ending
            )

    else:
        print("NRD image creation skipped; output directory already exists.")

NRD image creation skipped; output directory already exists.


NEN image creation: 100%|██████████| 391/391 [00:23<00:00, 16.44it/s]


In [34]:
# NReR combination (Nir, red edge, red)
ending = "NReR"

for index, path in enumerate(PATHS):

    input_dir = path[0]
    OUTPUT_DIR = path[1]
    dir = Path(str(OUTPUT_DIR) + "/" + ending)

    if not dir.exists():
        proc.set_input_path(input_dir / "image_tiles")

        for img_name in tqdm(sorted(os.listdir(proc.input_path)), desc="NEN image creation"):
            if "xml" in img_name:
                continue
            img_name_nir = img_name.replace(".tif", "_NIR.tif")
            img_name_er = img_name.replace(".tif", "_EXTEND_RED.tif")
            img_name_re = img_name.replace(".tif", "_REDEDGE.tif")
            proc.calculate_three_band_image(
                input_paths = ThreeBandInputPaths(
                    band1=input_dir / "extended_red" / img_name_er,
                    band2=input_dir / "rededge" / img_name_re,
                    band3=input_dir / "nir" / img_name_nir,
                ),
                output_path=dir,
                ending=ending
            )

    else:
        print("NReR image creation skipped; output directory already exists.")

NReR image creation skipped; output directory already exists.


NEN image creation:   0%|          | 0/391 [00:00<?, ?it/s]


RasterioIOError: /home/samuel/SDU/MasterThesis/Orthomosaics/small/processed_output/rededge/tile_0_0_REDEDGE.tif: No such file or directory

In [ ]:
# NReR combination (NDVI, red, green)
ending = "DRG"

for index, path in enumerate(PATHS):

    input_dir = path[0]
    OUTPUT_DIR = path[1]
    dir = Path(str(OUTPUT_DIR) + "/" + ending)

    if not dir.exists():
        proc.set_input_path(input_dir / "image_tiles")

        for img_name in tqdm(sorted(os.listdir(proc.input_path)), desc="NEN image creation"):
            if "xml" in img_name:
                continue
            img_name_ndvi = img_name.replace(".tif", "_ndvi.tif")
            img_name_er = img_name.replace(".tif", "_EXTEND_RED.tif")
            img_name_eg = img_name.replace(".tif", "_EXTEND_GREEN.tif")
            proc.calculate_three_band_image(
                input_paths = ThreeBandInputPaths(
                    band3=input_dir / "image_tiles_indeces" / "NDVI" / img_name_ndvi,
                    band1=input_dir / "extended_red" / img_name_er,
                    band2=input_dir / "extended_green" / img_name_eg,
                ),
                output_path=dir,
                ending=ending
            )

    else:
        print("DRG image creation skipped; output directory already exists.")

NEN image creation: 100%|██████████| 160/160 [00:14<00:00, 10.88it/s]
